# FIVES — cross-dataset evaluation, qualitative figure, in-domain training

`MODE = "crosseval"`: RFMiD-trained checkpoints (no retraining) tested on the FIVES test set
(200 images, 512×512) with full-image, FOV **and vessel** metrics + the qualitative figure on RFMiD. ≈ 45 min.

`MODE = "train"`: train + evaluate `fives_<EXPERIMENT>` for `SEEDS` on FIVES. E ≈ 1 h, L ≈ 1.2 h, P ≈ 2 h per seed.

**Setup (always):** GPU T4 · Internet ON · Add Input → FIVES dataset.
`crosseval` also needs: `retinal-disease-classification` + the Outputs of the 3 RFMiD training notebooks (`best.pt`).
Then **Save Version → Save & Run All (Commit)**. Download the zip from the Output tab (`fives_crosseval.zip` or `fives_train_<experiment>.zip`).

In [ ]:
MODE = "crosseval"              # crosseval | train
EXPERIMENT = "lorentz"         # train mode: euclidean | poincare | lorentz
SEEDS = [0, 1, 2]
REPO_URL = "https://github.com/hiayoub/poincare-vs-lorentz-denoising.git"
BRANCH = "main"

In [ ]:
from pathlib import Path
fives = sorted(Path("/kaggle/input").glob("**/train/Original"))
assert fives, "FIVES NOT ATTACHED: Add Input -> your FIVES dataset (must contain train/Original, test/Original)"
FIVES_ROOT = fives[0].parent.parent
n = {s: len(list((FIVES_ROOT / s / "Original").iterdir())) for s in ("train", "test")}
print("FIVES at", FIVES_ROOT, n)
assert n == {"train": 600, "test": 200}, f"unexpected FIVES size {n}"
if MODE == "crosseval":
    assert sorted(Path("/kaggle/input").glob("**/Training_Set")), "RFMiD NOT ATTACHED (needed for the qualitative figure)"
    ckpts = sorted(Path("/kaggle/input").glob("**/results/*_seed*/best.pt"))
    print(len(ckpts), "checkpoints:", sorted({c.parent.name for c in ckpts}))
    assert ckpts, "No checkpoints: attach the Output of the RFMiD training notebooks"

In [ ]:
import os, subprocess, sys
REPO = "/kaggle/working/repo"
if not os.path.isdir(REPO):
    subprocess.check_call(["git", "clone", "-q", "--depth", "1", "-b", BRANCH, REPO_URL, REPO])
os.chdir(REPO)
print(subprocess.check_output(["git", "log", "-1", "--format=commit %h  %s"]).decode())
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", "third_party/hypll", "-e", "."])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "lpips==0.1.4", "torchmetrics>=1.4", "pytest"])
assert Path("configs/eval_fives.yaml").exists(), "FIVES code not on GitHub yet: push repo_patch first"

In [ ]:
!python -m pytest tests -q
if _exit_code != 0:
    raise RuntimeError("tests failed")

## Run

In [ ]:
import shutil
OUT = Path("/kaggle/working/results")
OUT.mkdir(parents=True, exist_ok=True)
ROOT = f"data.root='{FIVES_ROOT}'"
if MODE == "crosseval":
    Path("configs/eval_fives_local.yaml").write_text(
        Path("configs/eval_fives.yaml").read_text().replace("root: null", f"root: '{FIVES_ROOT}'"))
    seen = set()
    for ck in ckpts:
        name = ck.parent.name
        if name in seen: continue
        seen.add(name)
        dst = OUT / name
        shutil.copytree(ck.parent, dst, dirs_exist_ok=True, ignore=shutil.ignore_patterns("last.pt"))
        !python -m hypdenoise.evaluate --run {dst} --data-config configs/eval_fives_local.yaml --tag fives
        if _exit_code != 0:
            raise RuntimeError(f"cross-eval failed for {name}")
    !python -m hypdenoise.compare --results {OUT} --reference euclidean --tag fives
    # Qualitative figure on RFMiD, heavy noise. Images = 50th and 75th percentile of the per-image
    # hyperbolic gain (PSNR_fov, 3 seeds) computed from the FOV re-evaluation: 235.png (+0.14 dB), 10.png (+0.69 dB)
    !python scripts/qualitative_figure.py --results {OUT} --seed 0 --images 235.png 10.png --out {OUT}/figures/qualitative_rfmid_heavy
else:
    for seed in SEEDS:
        !python -m hypdenoise.train --config configs/fives_{EXPERIMENT}.yaml --seed {seed} --set output_dir={OUT} {ROOT}
        if _exit_code != 0:
            raise RuntimeError(f"training failed for seed {seed}")
        !python -m hypdenoise.evaluate --run {OUT}/fives_{EXPERIMENT}_seed{seed}
        if _exit_code != 0:
            raise RuntimeError(f"evaluation failed for seed {seed}")

## Package the results (Output tab → zip)

In [ ]:
# crosseval: drop the copied checkpoints (already in the inputs); train: keep best.pt (needed later)
if MODE == "crosseval":
    for p in OUT.glob("*/best.pt"): p.unlink()
os.chdir("/kaggle/working")
shutil.rmtree(REPO, ignore_errors=True)
shutil.make_archive(f"/kaggle/working/fives_{MODE}{'' if MODE == 'crosseval' else '_' + EXPERIMENT}", "zip", OUT)
!ls -la /kaggle/working